In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
# Clean code
print(">>> STEP 1: READING DATA AND REMOVING OUTLIERS...")
# Read data
data = pd.read_csv('../data/train.csv')

# Remove the 2 outliers you found in the EDA step
data = data.drop(data[(data['GrLivArea'] > 4000) & (data['SalePrice'] < 300000)].index)

# Separate features (X) and target variable (y)
# Drop 'Id' because it doesn't have predictive power
X = data.drop(columns = ['Id', 'SalePrice'])
y = data['SalePrice']

# Identify numeric and categorical columns
numeric_features = X.select_dtypes(include = ['int64', 'float64']).columns
categorical_features = X.select_dtypes(include = ['object']).columns

print(f"Number of numeric features: {len(numeric_features)}")
print(f"Number of categorical features: {len(categorical_features)}")

print("\n>>> STEP 2: BUILDING PREPROCESSING PIPELINE...")
# 1. Numeric pipeline: Impute missing values with median -> Standard Scale
numeric_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', StandardScaler())
])

# 2. Categorical pipeline: Impute missing values with 'None' -> One-hot encoding
categorical_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'constant', fill_value = 'None')),
    ('onehot', OneHotEncoder(handle_unknown = 'ignore'))
])

# Combine numeric and categorical transformers into one master preprocessor
preprocessor = ColumnTransformer(
    transformers = [
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

>>> STEP 1: READING DATA AND REMOVING OUTLIERS...
Number of numeric features: 36
Number of categorical features: 43

>>> STEP 2: BUILDING PREPROCESSING PIPELINE...


In [5]:
# Training model
# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

print("\n>>> STEP 3: TRAINING AND EVALUATING MODELS...")
# Define 4 models to compare
models = {
    "1. Linear Regression": LinearRegression(),
    "2. Ridge Regression (L2)": Ridge(alpha = 10.0),
    "3. Lasso Regression (L1)": Lasso(alpha = 100.0, max_iter = 10000),
    "4. Random Forest Regressor": RandomForestRegressor(n_estimators = 100, random_state = 42)
}

# Loop through each model, train, and evaluate
for name, model in models.items():
    # Create the final pipeline combining preprocessing and the model
    clf = Pipeline(steps=[('preprocessor', preprocessor),
                          ('model', model)])
    
    # Train the model
    clf.fit(X_train, y_train)
    
    # Predict on the test set
    y_pred = clf.predict(X_test)
    
    # Calculate evaluation metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(f"{name}")
    print(f"   Mean Absolute Error (MAE): ${mae:,.0f}")
    print(f"   Root Mean Squared Error (RMSE): ${rmse:,.0f}")
    print(f"   R2 Score                 : {r2 * 100:.2f}%\n")


>>> STEP 3: TRAINING AND EVALUATING MODELS...
1. Linear Regression
   Mean Absolute Error (MAE): $20,821
   Root Mean Squared Error (RMSE): $60,079
   R2 Score                 : 34.66%

2. Ridge Regression (L2)
   Mean Absolute Error (MAE): $16,880
   Root Mean Squared Error (RMSE): $23,247
   R2 Score                 : 90.22%

3. Lasso Regression (L1)
   Mean Absolute Error (MAE): $16,366
   Root Mean Squared Error (RMSE): $23,029
   R2 Score                 : 90.40%

4. Random Forest Regressor
   Mean Absolute Error (MAE): $16,995
   Root Mean Squared Error (RMSE): $24,376
   R2 Score                 : 89.24%



In [6]:
# Hyperparameter tuning
print("\n>>> STEP 4: HYPERPARAMETER TUNING FOR RANDOM FOREST...")

# Create a specific pipeline for Random Forest
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state = 42))
])

# Define the parameter grid to test
# Note: Use 'model__' prefix since the model is inside a Pipeline
param_grid = {
    'model__n_estimators': [100, 200, 300],        # Test different numbers of trees
    'model__max_depth': [None, 10, 20],            # Test different maximum depths
    'model__min_samples_split': [2, 5, 10]         # Minimum samples required to split an internal node
}

# Initialize GridSearchCV
# cv=5: 5-fold cross-validation
# n_jobs=-1: Use all available CPU cores for faster training
grid_search = GridSearchCV(rf_pipeline, param_grid, cv = 5, 
                           scoring = 'r2', n_jobs = -1, verbose = 1)

# START GRID SEARCH (This will take a few minutes)
grid_search.fit(X_train, y_train)

# Get the best estimator found by GridSearchCV
best_rf_model = grid_search.best_estimator_

print("\n>>> TUNING COMPLETE!")
print(f"Best Parameters: {grid_search.best_params_}")

# Evaluate the best model on the test set
y_pred_best = best_rf_model.predict(X_test)
best_mae = mean_absolute_error(y_test, y_pred_best)
best_rmse = np.sqrt(mean_squared_error(y_test, y_pred_best))
best_r2 = r2_score(y_test, y_pred_best)

print("OPTIMIZED RANDOM FOREST RESULTS:")
print(f"   Mean Absolute Error (MAE): ${best_mae:,.0f}")
print(f"   Root Mean Squared Error (RMSE): ${best_rmse:,.0f}")
print(f"   R2 Score                 : {best_r2 * 100:.2f}%")


>>> STEP 4: HYPERPARAMETER TUNING FOR RANDOM FOREST...
Fitting 5 folds for each of 27 candidates, totalling 135 fits

>>> TUNING COMPLETE!
Best Parameters: {'model__max_depth': 20, 'model__min_samples_split': 2, 'model__n_estimators': 300}
OPTIMIZED RANDOM FOREST RESULTS:
   Mean Absolute Error (MAE): $16,774
   Root Mean Squared Error (RMSE): $24,053
   R2 Score                 : 89.53%


In [10]:
# FInal model
print(">>> STEP 5: SAVING THE FINAL MODEL...")

# 1. Re-initialize the exact Pipeline of the winning model (Lasso)
final_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Lasso(alpha = 100.0, max_iter = 10000))
])

# 2. Retrain the model on the entire dataset (X, y) to maximize learning
final_model.fit(X, y)

# 3. Export the trained model to a file
joblib.dump(final_model, '../house_price_model.pkl')
print("Export successful! The file 'house_price_model.pkl' has been created.")

>>> STEP 5: SAVING THE FINAL MODEL...
Export successful! The file 'house_price_model.pkl' has been created.
